<a href="https://colab.research.google.com/github/nirlondev/Algoritms/blob/main/titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
import pandas as pd
import re
from google.colab import files
from io import BytesIO
from datetime import datetime

In [45]:

uploaded = files.upload()
filename = list(uploaded.keys())[0]

content = uploaded[filename].decode('utf-8-sig')
lines = content.strip().split('\n')

lines = [line for line in lines if line.strip()]

header = lines[0].replace('"', '').strip().split('\t')
print("Заголовок столбцов:", header)

data = []
for line in lines[1:]:

    values = line.replace('"', '').strip().split('\t')

    if len(values) < len(header):
        values += [''] * (len(header) - len(values))
    elif len(values) > len(header):
        values = values[:len(header)]
    data.append(values)

df = pd.DataFrame(data, columns=header)

numeric_cols = ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

display(df.head())
print("Столбцы:", df.columns.tolist())

Saving titanik_full_data.csv to titanik_full_data (14).csv
Заголовок столбцов: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,,S


Столбцы: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [46]:

df.columns = df.columns.str.replace('"', '')

for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.replace('"', '')

df['Survived'] = pd.to_numeric(df['Survived'], errors='coerce')

print("Данные очищены. Информация о DataFrame:")
print(df.info())

Данные очищены. Информация о DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Survived     1309 non-null   int64  
 2   Pclass       1309 non-null   int64  
 3   Name         1309 non-null   object 
 4   Sex          1309 non-null   object 
 5   Age          1046 non-null   float64
 6   SibSp        1309 non-null   int64  
 7   Parch        1309 non-null   int64  
 8   Ticket       1309 non-null   object 
 9   Fare         1308 non-null   float64
 10  Cabin        1309 non-null   object 
 11  Embarked     1309 non-null   object 
dtypes: float64(2), int64(5), object(5)
memory usage: 122.8+ KB
None


In [47]:

def extractSurname(name):
    if pd.isna(name):
        return ''
    return name.split(',')[0].strip()

def countVowels(text):
    if pd.isna(text):
        return 0
    vowels = set('aeiou')
    return sum(1 for ch in str(text).lower() if ch in vowels)

def isEvenLength(text):
    if pd.isna(text):
        return False
    return len(str(text)) % 2 == 0


In [48]:

df['Surname'] = df['Name'].apply(extractSurname)
df['SurnameLength'] = df['Surname'].apply(len)

df['VowelCount'] = df['Name'].apply(countVowels)


df['EvenNameLength'] = df['Name'].apply(isEvenLength)

display(df[['Name', 'Surname', 'SurnameLength', 'VowelCount', 'EvenNameLength']].head(10))

,Name,Surname,SurnameLength,VowelCount,EvenNameLength
0,"Braund, Mr. Owen Harris",Braund,6,6,False
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Cumings,7,11,False
2,"Heikkinen, Miss. Laina",Heikkinen,9,8,True
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Futrelle,8,12,True
4,"Allen, Mr. William Henry",Allen,5,6,True
5,"Moran, Mr. James",Moran,5,4,True
6,"McCarthy, Mr. Timothy J",McCarthy,8,3,False
7,"Palsson, Master. Gosta Leonard",Palsson,7,9,True
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",Johnson,7,13,False
9,"Nasser, Mrs. Nicholas (Adele Achem)",Nasser,6,10,False


In [49]:
totalPassengers = len(df)
print(f" Всего пассажиров в списке: {totalPassengers}")

 Всего пассажиров в списке: 1309


In [50]:
survivedWomen = df[(df['Sex'] == 'female') & (df['Survived'] == 1)].shape[0]
survivedMen = df[(df['Sex'] == 'male') & (df['Survived'] == 1)].shape[0]
print(f"   Выжило женщин: {survivedWomen}")
print(f"   Выжило мужчин: {survivedMen}")

   Выжило женщин: 385
   Выжило мужчин: 109


In [51]:

maskLong = df['SurnameLength'] >= 8

totalLong = maskLong.sum()
survivedLong = df[maskLong & (df['Survived'] == 1)].shape[0]
probAll = survivedLong / totalLong if totalLong > 0 else 0

maskFemaleLong = (df['Sex'] == 'female') & maskLong
totalFemaleLong = maskFemaleLong.sum()
survivedFemaleLong = df[maskFemaleLong & (df['Survived'] == 1)].shape[0]
probFemale = survivedFemaleLong / totalFemaleLong if totalFemaleLong > 0 else 0

maskMaleLong = (df['Sex'] == 'male') & maskLong
totalMaleLong = maskMaleLong.sum()
survivedMaleLong = df[maskMaleLong & (df['Survived'] == 1)].shape[0]
probMale = survivedMaleLong / totalMaleLong if totalMaleLong > 0 else 0

print("3. Вероятность выжить, если фамилия длиной 8 и более символов:")
print(f"   Для всех: {probAll:.3f} ({survivedLong}/{totalLong})")
print(f"   Для женщин: {probFemale:.3f} ({survivedFemaleLong}/{totalFemaleLong})")
print(f"   Для мужчин: {probMale:.3f} ({survivedMaleLong}/{totalMaleLong})")

3. Вероятность выжить, если фамилия длиной 8 и более символов:
   Для всех: 0.362 (135/373)
   Для женщин: 0.795 (93/117)
   Для мужчин: 0.164 (42/256)


In [52]:
maxLen = df['SurnameLength'].max()
maxRows = df[df['SurnameLength'] == maxLen]

print(f"4. Максимальная длина фамилии: {maxLen} символов.")
print("   Пассажиры с такой фамилией:")
display(maxRows[['Name', 'Surname', 'Survived']])

survivedMax = maxRows[maxRows['Survived'] == 1]
if not survivedMax.empty:
    print("    Среди них есть выжившие.")
else:
    print("    Среди них нет выживших.")

4. Максимальная длина фамилии: 22 символов.
   Пассажиры с такой фамилией:


,Name,Surname,Survived
430,"Bjornstrom-Steffansson, Mr. Mauritz Hakan",Bjornstrom-Steffansson,1
444,"Johannesen-Bratthammer, Mr. Bernt",Johannesen-Bratthammer,1


    Среди них есть выжившие.


In [53]:
pivotVowels = df.groupby(['Pclass', 'Sex'])['VowelCount'].mean().unstack()
print("5. Среднее количество гласных в полном имени по классу и полу:")
print(pivotVowels)

counts = df.groupby(['Pclass', 'Sex']).size().unstack()
print("\n   Количество пассажиров в группах:")
print(counts)

classStats = df.groupby('Pclass')['VowelCount'].agg(['mean', 'std', 'count'])
print("\n   Общая статистика по классам:")
print(classStats)

5. Среднее количество гласных в полном имени по классу и полу:
Sex        female      male
Pclass                     
1       10.395833  6.564246
2        9.150943  6.274854
3        9.074074  6.040568

   Количество пассажиров в группах:
Sex     female  male
Pclass              
1          144   179
2          106   171
3          216   493

   Общая статистика по классам:
            mean       std  count
Pclass                           
1       8.272446  3.408987    323
2       7.375451  3.080416    277
3       6.964739  2.865664    709


In [54]:
diedClass1 = df[(df['Pclass'] == 1) & (df['Survived'] == 0)].shape[0]
diedClass3 = df[(df['Pclass'] == 3) & (df['Survived'] == 0)].shape[0]
print(f"6. Погибло в 1-ом классе: {diedClass1}")
print(f"   Погибло в 3-ем классе: {diedClass3}")

6. Погибло в 1-ом классе: 137
   Погибло в 3-ем классе: 518


In [55]:
portStats = df.groupby('Embarked')['Survived'].agg(['count', 'sum'])
portStats.columns = ['Всего', 'Выжило']
portStats['Доля выживших'] = portStats['Выжило'] / portStats['Всего']
print("7. Статистика выживших по порту посадки:")
print(portStats)

7. Статистика выживших по порту посадки:
          Всего  Выжило  Доля выживших
Embarked                              
              2       2       1.000000
C           270     133       0.492593
Q           123      54       0.439024
S           914     305       0.333698


In [56]:
meanAge = df['Age'].mean()
missingAge = df['Age'].isna().sum()
print(f"8. Средний возраст пассажиров: {meanAge:.2f} лет")
print(f"   Пропущено данных о возрасте: {missingAge}")

8. Средний возраст пассажиров: 29.88 лет
   Пропущено данных о возрасте: 263


In [57]:
evenNameDf = df[df['EvenNameLength'] == True].copy()
print(f"9. Пассажиры с чётным количеством символов в имени (всего {len(evenNameDf)}):")
display(evenNameDf)

9. Пассажиры с чётным количеством символов в имени (всего 635):


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Surname,SurnameLength,VowelCount,EvenNameLength
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,,S,Heikkinen,9,8,True
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Futrelle,8,12,True
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,,S,Allen,5,6,True
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,,Q,Moran,5,4,True
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,,S,Palsson,7,9,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1303,1304,1,3,"Henriksson, Miss. Jenny Lovisa",female,28.0,0,0,347086,7.7750,,S,Henriksson,10,8,True
1304,1305,0,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,,S,Spector,7,4,True
1305,1306,1,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C,Oliva y Ocana,13,11,True
1306,1307,0,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,,S,Saether,7,8,True


In [58]:
outputFileExcel = 'titanic_processed.xlsx'
outputFileCsv = 'titanic_processed.csv'

df.to_excel(outputFileExcel, index=False)
df.to_csv(outputFileCsv, index=False, encoding='utf-8-sig')

files.download(outputFileExcel)
files.download(outputFileCsv)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>